# 01 Provider Routing and Fallback (LiteLLM, 2026)

## What This Lesson Is
Implement explicit provider routing and fallback logic instead of relying on hidden defaults.

## Scientific Lens
- Concept: Reliability through ordered fallback chains
- Measure: Request success rate when primary provider fails
- Validity Limit: Simulated outages do not capture provider-wide correlated failures.


## How It Works
1. Define deterministic route selection and fallback order.
2. Exercise failure cases with synthetic provider states.
3. Run a real LiteLLM call with a fallback path.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Default OpenAI model:", os.getenv("OPENAI_MODEL", "gpt-4.1-mini"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
providers = {
    "openai_primary": {"up": False, "priority": 1},
    "openai_secondary": {"up": True, "priority": 2},
    "local_backup": {"up": True, "priority": 3},
}

order = sorted(providers.items(), key=lambda kv: kv[1]["priority"])
selected = None
for name, cfg in order:
    if cfg["up"]:
        selected = name
        break

print("selected provider:", selected)
assert selected == "openai_secondary"


In [ ]:
# Live Demo
import os

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live LiteLLM call: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live LiteLLM call: OPENAI_API_KEY not set.")
    else:
        prompt = "One sentence: why fallback routing matters in production."
        candidates = ["openai/gpt-4.1-mini", "openai/gpt-4o-mini"]
        final_text = None
        for model in candidates:
            try:
                r = completion(model=model, messages=[{"role": "user", "content": prompt}], api_key=api_key, timeout=20)
                final_text = r.choices[0].message.content.strip()
                print("model used:", model)
                break
            except Exception as exc:
                print(f"fallback from {model}: {exc}")
        print(final_text)
        assert final_text


## Applied Labs
1. Flip provider availability states and verify selection order changes as expected.
2. Introduce a "hard-fail" provider and confirm it does not block downstream fallback.
3. Record provider-level success counters and compare before/after fallback tuning.

## Validation Checklist
- Primary failure does not terminate the whole request path.
- Fallback order is explicit and deterministic.
- Live run prints which model actually served the response.

## Further Reading
- [LiteLLM Docs - Routing](https://docs.litellm.ai/docs/routing)
- [LiteLLM Docs - Completion](https://docs.litellm.ai/docs/completion)
- [Google SRE - Handling Overload](https://sre.google/sre-book/handling-overload/)
